In [8]:
import os
import re
import glob
import numpy as np
import pandas as pd
from pathlib import Path
import shutil
from tqdm import tqdm


In [15]:
# Pandas :: view ALL
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [18]:
# Pandas :: Default view
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

In [2]:
# Defines
def analyse_dataframe(df):
    nan_cnt = df.isnull().sum()
    nan_rto = df.isnull().mean()*100
    unique_vals_cnt = df.nunique()

    ana_result = pd.DataFrame({
        'NaN Counts' : nan_cnt
        , 'NaN Ratio (%)' : nan_rto
        , 'Unique Values' : unique_vals_cnt
    })
    return ana_result


In [9]:
## PRE
out = r"D:\JHAhn\files"
src_root = Path(r'D:\JHAhn\8641_conductivity\DTI_ALPS\DTI_ALPS_Result\2_Dementia')
dst_root = Path(r'E:\MR\Dataset')

In [4]:
sheets = pd.read_excel(os.path.join(ALPS_DIR, "Auto_ALPS_index_19-24_0731.xlsx")
                       , sheet_name=["Sheet1", "8641", "8641_Dementia"]
                      )
dfS1 = sheets["Sheet1"]    # base
df8641 = sheets["8641"]    # CN
dfDem = sheets["8641_Dementia"]    # Dementia


def clean_pad_pid(df):
    """PID 컬럼을 8자리 0-패딩 문자열로 변환 (결측값은 그대로)"""
    if "PID" not in df.columns:
        return df

    mask = df["PID"].notna()                                 # 결측이 아닌 행
    df.loc[mask, "PID"] = (
        df.loc[mask, "PID"]
          .astype(str)                                        # → 문자열
          .str.replace(r"\.0$", "", regex=True)               # ① 끝 '.0' 제거
          .str.replace(r"[^\d]", "", regex=True)              # ② 혹시 남은 비숫자 제거
          .str.zfill(8)                                       # ③ 8자리 0-패딩
    ).astype("string")                                        # ④ dtype 문자열 고정
    return df

for df in (dfS1, df8641, dfDem):
    clean_pad_pid(df)


print(dfS1.shape, df8641.shape, dfDem.shape, '\n')
print(dfS1, df8641, dfDem)

(206, 52) (58, 105) (209, 105) 

     건대분석정보                 인원    DDX     AD_MWI_Code    미수코드  \
0    NL(39)                NaN    NaN             NaN     NaN   
1       Yea                NaN     NL     EPT2022AD40     NaN   
2        No                NaN     NL     EPT2023AD09     NaN   
3       Yes                  1     NL        MWI_CN01  8641.0   
4       Yes                  2     NL        MWI_CN04  8641.0   
..      ...                ...    ...             ...     ...   
201     Yes             9탈락3-2  NaMCI  MWI_CN66(ver2)  8641.0   
202     Yes            10탈락3-3  NaMCI  MWI_CN67(ver2)  8641.0   
203     Yes            13탈락3-4  NaMCI  MWI_CN70(ver2)  8641.0   
204     Yes      탈락-infarction  NaMCI   MREPT2020AD10     NaN   
205     Yes  탈락-venous anomaly  NaMCI   MREPT2020AD27     NaN   

                 논문사용       PID Name   Age ChiQSM  ... MR_3DT1_TFE  \
0                 NaN       NaN  NaN   NaN    NaN  ...         NaN   
1          2022 상반기영상  09463703  유길종  74.0  CN

In [5]:
dfS1

,건대분석정보,인원,DDX,AD_MWI_Code,미수코드,논문사용,PID,Name,Age,ChiQSM,...,MR_3DT1_TFE,MR_3DGRASE_MWI,MR_3DGRASE_REF,MREPT_TSE,NODDI,MREPT_3DbFFE,Unnamed: 48,conductivity,MWF,Chi
0,NL(39),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Yea,NaN,NL,EPT2022AD40,NaN,2022 상반기영상,09463703,유길종,74.0,CNL02,...,O,O,O,O,NODDI32,X,NaN,O,O,O
2,No,NaN,NL,EPT2023AD09,NaN,NaN,08787368,이송웅,81.0,CNL03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
3,Yes,1,NL,MWI_CN01,8641.0,"HFC,LFC사용",00145616,강난희,70.0,CNL04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
4,Yes,2,NL,MWI_CN04,8641.0,HFC 사용,02716077,송승숙,62.0,CNL05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,Yes,9탈락3-2,NaMCI,MWI_CN66(ver2),8641.0,newMWI_CN 27명 추가,00766581,조수연,71.0,CNM25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
202,Yes,10탈락3-3,NaMCI,MWI_CN67(ver2),8641.0,newMWI_CN 27명 추가,09577263,김오심,82.0,CNM26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
203,Yes,13탈락3-4,NaMCI,MWI_CN70(ver2),8641.0,newMWI_CN 27명 추가,02202449,이청자,80.0,CNM27,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
204,Yes,탈락-infarction,NaMCI,MREPT2020AD10,NaN,NaN,08276845,조영자,73.0,CNM29,...,O,O,O,O,NODDI32,X,NaN,NaN,탈락-infarction,NaN


In [6]:
def move_to_sgds(df):
    # SGDS 칼럼이 없으면 먼저 만들기
    if "SGDS" not in df.columns:
        df["SGDS"] = pd.NA

    # ① SGDS + 공백
    pat1 = re.compile(r"^SGDS\s*(.+)$")
    mask1 = df["NPI"].astype(str).str.match(pat1, na=False)
    df.loc[mask1, "SGDS"] = df.loc[mask1, "NPI"].str.replace(pat1, r"\1", regex=True).str.strip()
    df.loc[mask1, "NPI"]  = pd.NA

    # ② SGDS=
    pat2 = re.compile(r"^SGDS=(.+)$")
    mask2 = df["NPI"].astype(str).str.match(pat2, na=False)
    df.loc[mask2, "SGDS"] = df.loc[mask2, "NPI"].str.replace(pat2, r"\1", regex=True).str.strip()
    df.loc[mask2, "NPI"]  = pd.NA

    # ③ *(normal)
    pat3 = re.compile(r"^(.+?)\s*\(normal\)$")
    mask3 = df["NPI"].astype(str).str.match(pat3, na=False)
    df.loc[mask3, "SGDS"] = df.loc[mask3, "NPI"].str.replace(pat3, r"\1", regex=True).str.strip()
    df.loc[mask3, "NPI"]  = pd.NA

    return df

dfS1 = move_to_sgds(dfS1)

dfS1[['NPI', 'SGDS']]

,NPI,SGDS
0,NaN,NaN
1,0/144,NaN
2,NaN,NaN
3,0/44,NaN
4,3/144,NaN
...,...,...
201,NaN,<NA>
202,<NA>,1/15
203,NaN,<NA>
204,0/144,<NA>


In [7]:
analyse_dataframe(dfS1)

,NaN Counts,NaN Ratio (%),Unique Values
건대분석정보,4,1.941748,8
인원,101,49.029126,46
DDX,9,4.368932,5
AD_MWI_Code,9,4.368932,197
미수코드,136,66.019417,1
논문사용,55,26.699029,14
PID,9,4.368932,195
Name,9,4.368932,196
Age,9,4.368932,36
ChiQSM,14,6.796117,192


In [8]:
def to_pdi(x):
    parts = str(x).split("_")
    return "_".join(parts[:-2]) if len(parts) >= 3 else parts[0]

for df in (df8641, dfDem):
    if "PDI" not in df.columns:
        df["PDI"] = df["PID"].map(to_pdi)

In [9]:
df = dfS1
for column in df.columns:
    print(f"Col name: {column}")
    print(f"Data type: {df[column].dtype}")
    print(f"Unique values: {len(df[column].unique())}")
    print(df[column].unique())
    print('-'*40)

Col name: 건대분석정보
Data type: object
Unique values: 9
['NL(39)' 'Yea' 'No' 'Yes' nan 'SaMCI(41)' 'MaMCI(36)' 'AD (35)'
 'NaMCI(20)']
----------------------------------------
Col name: 인원
Data type: object
Unique values: 47
[nan 1 2 3 4 5 10 17 18 19 20 21 22 8 9 11 14 15 25 6 12 13 '탈락' 27 24 16
 7 '점수미달' 'MREPT2019AD10' 23 28 '탈락-hydrocephalus' 'MREPT2019AD09'
 'MREPT2019AD13' '탈락-SAH' '4탈락-' 'MREPT2019AD11' '7탈락1-2' '12탈락1-5'
 '14탈락1-7' '7탈락2-1' '19탈락2-2' '9탈락3-2' '10탈락3-3' '13탈락3-4' '탈락-infarction'
 '탈락-venous anomaly']
----------------------------------------
Col name: DDX
Data type: object
Unique values: 6
[nan 'NL' 'SaMCI' 'MaMCI' 'AD' 'NaMCI']
----------------------------------------
Col name: AD_MWI_Code
Data type: object
Unique values: 198
[nan 'EPT2022AD40' 'EPT2023AD09' 'MWI_CN01' 'MWI_CN04' 'MWI_CN05'
 'MWI_CN06' 'MWI_CN07' 'MWI_CN12' 'MWI_CN19' 'MWI_CN20' 'MWI_CN21'
 'MWI_CN22' 'MWI_CN23' 'MWI_CN24' 'MWI_CN33' 'MWI_CN35' 'MWI_CN38'
 'MWI_CN39' 'MWI_CN40' 'MWI_CN41' 'MWI_CN44

In [10]:
# Cols to add
cols_8641 = ["PDI", "PID"
             , "TIV", "global GMV", "global WMV", "global CSFV", "NPI", "SGDS"
             , "MMSE", "CDR", "CDRSB", "GDS", "Barthel A DL", "Instrumental A DL"
             , 'LT_ALL_Dxx_proj', 'LT_ALL_Dyy_proj', 'LT_ALL_Dzz_proj', 'LT_ALL_Dxx_assoc'
             , 'LT_ALL_Dyy_assoc', 'LT_ALL_Dzz_assoc', 'LT_ALL_ALPS index','LT_b800_Dxx_proj'
             , 'LT_b800_Dyy_proj', 'LT_b800_Dzz_proj', 'LT_b800_Dxx_assoc', 'LT_b800_Dyy_assoc'
             , 'LT_b800_Dzz_assoc', 'LT_b800_ALPS index', 'LT_b2000_Dxx_proj', 'LT_b2000_Dyy_proj'
             , 'LT_b2000_Dzz_proj', 'LT_b2000_Dxx_assoc', 'LT_b2000_Dyy_assoc', 'LT_b2000_Dzz_assoc'
             , 'LT_b2000_ALPS index', 'RT_ALL_Dxx_proj', 'RT_ALL_Dyy_proj', 'RT_ALL_Dzz_proj'
             , 'RT_ALL_Dxx_assoc', 'RT_ALL_Dyy_assoc', 'RT_ALL_Dzz_assoc', 'RT_ALL_ALPS index'
             , 'RT_b800_Dxx_proj', 'RT_b800_Dyy_proj', 'RT_b800_Dzz_proj', 'RT_b800_Dxx_assoc'
             , 'RT_b800_Dyy_assoc', 'RT_b800_Dzz_assoc', 'RT_b800_ALPS index', 'RT_b2000_Dxx_proj'
             , 'RT_b2000_Dyy_proj', 'RT_b2000_Dzz_proj', 'RT_b2000_Dxx_assoc', 'RT_b2000_Dyy_assoc'
             , 'RT_b2000_Dzz_assoc', 'RT_b2000_ALPS index'
             , 'HFC_LT_Amygdala', 'IC_LT_Amygdala', 'EC_LT_Amygdala', 'HFC_LT_Hippocampus', 'IC_LT_Hippocampus'
             , 'EC_LT_Hippocampus', 'HFC_LT_Insula', 'IC_LT_Insula', 'EC_LT_Insula', 'HFC_LT_MTG', 'IC_LT_MTG'
             , 'EC_LT_MTG', 'HFC_LT_ParaHippocampus', 'IC_LT_ParaHippocampus', 'EC_LT_ParaHippocampus'
             , 'HFC_LT_Thalamus', 'IC_LT_Thalamus', 'EC_LT_Thalamus', 'HFC_RT_Amygdala', 'IC_RT_Amygdala'
             , 'EC_RT_Amygdala', 'HFC_RT_Hippocampus', 'IC_RT_Hippocampus', 'EC_RT_Hippocampus'
             , 'HFC_RT_Insula', 'IC_RT_Insula', 'EC_RT_Insula', 'HFC_RT_MTG', 'IC_RT_MTG', 'EC_RT_MTG'
             , 'HFC_RT_ParaHippocampus', 'IC_RT_ParaHippocampus', 'EC_RT_ParaHippocampus'
             , 'HFC_RT_Thalamus', 'IC_RT_Thalamus', 'EC_RT_Thalamus', 'HFC_Whole_CC', 'IC_Whole_CC', 'EC_Whole_CC'
             , 'HFC_Whole_Precuneus', 'IC_Whole_Precuneus', 'EC_Whole_Precuneus'
            ]

cols_8641_dem = ["PDI", "PID"
                 , "TIV", "global GMV", "global WMV", "global CSFV", "NPI", "SGDS"
                 , "MMSE", "CDR", "CDRSB", "GDS", "Barthel A DL", "Instrumental A DL"
                 , 'LT_ALL_Dxx_proj', 'LT_ALL_Dyy_proj', 'LT_ALL_Dzz_proj', 'LT_ALL_Dxx_assoc'
                 , 'LT_ALL_Dyy_assoc', 'LT_ALL_Dzz_assoc', 'LT_ALL_ALPS index','LT_b800_Dxx_proj'
                 , 'LT_b800_Dyy_proj', 'LT_b800_Dzz_proj', 'LT_b800_Dxx_assoc', 'LT_b800_Dyy_assoc'
                 , 'LT_b800_Dzz_assoc', 'LT_b800_ALPS index', 'LT_b2000_Dxx_proj', 'LT_b2000_Dyy_proj'
                 , 'LT_b2000_Dzz_proj', 'LT_b2000_Dxx_assoc', 'LT_b2000_Dyy_assoc', 'LT_b2000_Dzz_assoc'
                 , 'LT_b2000_ALPS index', 'RT_ALL_Dxx_proj', 'RT_ALL_Dyy_proj', 'RT_ALL_Dzz_proj'
                 , 'RT_ALL_Dxx_assoc', 'RT_ALL_Dyy_assoc', 'RT_ALL_Dzz_assoc', 'RT_ALL_ALPS index'
                 , 'RT_b800_Dxx_proj', 'RT_b800_Dyy_proj', 'RT_b800_Dzz_proj', 'RT_b800_Dxx_assoc'
                 , 'RT_b800_Dyy_assoc', 'RT_b800_Dzz_assoc', 'RT_b800_ALPS index', 'RT_b2000_Dxx_proj'
                 , 'RT_b2000_Dyy_proj', 'RT_b2000_Dzz_proj', 'RT_b2000_Dxx_assoc', 'RT_b2000_Dyy_assoc'
                 , 'RT_b2000_Dzz_assoc', 'RT_b2000_ALPS index'
                 , 'HFC_LT_Amygdala', 'IC_LT_Amygdala', 'EC_LT_Amygdala', 'HFC_LT_Hippocampus', 'IC_LT_Hippocampus'
                 , 'EC_LT_Hippocampus', 'HFC_LT_Insula', 'IC_LT_Insula', 'EC_LT_Insula', 'HFC_LT_MTG', 'IC_LT_MTG'
                 , 'EC_LT_MTG', 'HFC_LT_ParaHippocampus', 'IC_LT_ParaHippocampus', 'EC_LT_ParaHippocampus'
                 , 'HFC_LT_Thalamus', 'IC_LT_Thalamus', 'EC_LT_Thalamus', 'HFC_RT_Amygdala', 'IC_RT_Amygdala'
                 , 'EC_RT_Amygdala', 'HFC_RT_Hippocampus', 'IC_RT_Hippocampus', 'EC_RT_Hippocampus'
                 , 'HFC_RT_Insula', 'IC_RT_Insula', 'EC_RT_Insula', 'HFC_RT_MTG', 'IC_RT_MTG', 'EC_RT_MTG'
                 , 'HFC_RT_ParaHippocampus', 'IC_RT_ParaHippocampus', 'EC_RT_ParaHippocampus'
                 , 'HFC_RT_Thalamus', 'IC_RT_Thalamus', 'EC_RT_Thalamus', 'HFC_Whole_CC', 'IC_Whole_CC', 'EC_Whole_CC'
                 , 'HFC_Whole_Precuneus', 'IC_Whole_Precuneus', 'EC_Whole_Precuneus'
                ]

priority_cols = ["TIV", "global GMV", "global WMV", "global CSFV", "NPI", "SGDS"
                 , "MMSE", "CDR", "CDRSB", "GDS", "Barthel A DL", "Instrumental A DL"
                ]
skip_cols = set(priority_cols + ["PID", "PDI"])

add_cols_8641 = [c for c in cols_8641 if c not in skip_cols]
add_cols_8641_dem = [c for c in cols_8641_dem if c not in skip_cols]

In [11]:
def fill_missing(base, donor, cols, base_key, donor_key):
    ref = donor[[donor_key] + cols].copy().rename(columns={donor_key: base_key})
    merged = base[[base_key] + cols].merge(ref, on=base_key, how="left",
                                           suffixes=("", "_donor"))
    for col in cols:
        base[col] = base[col].fillna(merged[f"{col}_donor"])
    return base

dfS1 = fill_missing(dfS1, df8641, priority_cols, "PID", "PID")
dfS1 = fill_missing(dfS1, dfDem,  priority_cols, "PID", "PID")
print(dfS1[["TIV", "global GMV", "global WMV", "global CSFV", "NPI", "MMSE"
            , "CDR", "CDRSB", "GDS", "Barthel A DL", "Instrumental A DL"]]
     )

dfS1 = fill_missing(dfS1, df8641, priority_cols, "AD_MWI_Code", "PDI")
dfS1 = fill_missing(dfS1, dfDem,  priority_cols, "AD_MWI_Code", "PDI")

dfS1

             TIV  global GMV  global WMV  global CSFV     NPI  MMSE  CDR  \
0            NaN         NaN         NaN          NaN     NaN   NaN  NaN   
1    1550.469470  493.238319  412.905559   639.431894   0/144  29.0  0.0   
2            NaN         NaN         NaN          NaN     NaN  30.0  0.5   
3    1304.359887  499.301540  344.757090   457.000531    0/44  30.0  0.0   
4    1431.074737  517.657305  405.560720   505.216261   3/144  30.0  0.0   
..           ...         ...         ...          ...     ...   ...  ...   
201          NaN         NaN         NaN          NaN     NaN  27.0  0.5   
202          NaN         NaN         NaN          NaN     NaN  21.0  0.5   
203          NaN         NaN         NaN          NaN     NaN  28.0  0.5   
204  1275.839545  457.229861  392.933175   425.676509   0/144  28.0  0.5   
205          NaN         NaN         NaN          NaN  27/144  22.0  0.5   

     CDRSB  GDS Barthel A DL Instrumental A DL  
0      NaN  NaN          NaN          

,건대분석정보,인원,DDX,AD_MWI_Code,미수코드,논문사용,PID,Name,Age,ChiQSM,...,MR_3DGRASE_MWI,MR_3DGRASE_REF,MREPT_TSE,NODDI,MREPT_3DbFFE,Unnamed: 48,conductivity,MWF,Chi,SGDS
0,NL(39),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Yea,NaN,NL,EPT2022AD40,NaN,2022 상반기영상,09463703,유길종,74.0,CNL02,...,O,O,O,NODDI32,X,NaN,O,O,O,0/15
2,No,NaN,NL,EPT2023AD09,NaN,NaN,08787368,이송웅,81.0,CNL03,...,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O,NaN
3,Yes,1,NL,MWI_CN01,8641.0,"HFC,LFC사용",00145616,강난희,70.0,CNL04,...,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O,NaN
4,Yes,2,NL,MWI_CN04,8641.0,HFC 사용,02716077,송승숙,62.0,CNL05,...,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,Yes,9탈락3-2,NaMCI,MWI_CN66(ver2),8641.0,newMWI_CN 27명 추가,00766581,조수연,71.0,CNM25,...,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O,NaN
202,Yes,10탈락3-3,NaMCI,MWI_CN67(ver2),8641.0,newMWI_CN 27명 추가,09577263,김오심,82.0,CNM26,...,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O,1/15
203,Yes,13탈락3-4,NaMCI,MWI_CN70(ver2),8641.0,newMWI_CN 27명 추가,02202449,이청자,80.0,CNM27,...,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O,NaN
204,Yes,탈락-infarction,NaMCI,MREPT2020AD10,NaN,NaN,08276845,조영자,73.0,CNM29,...,O,O,O,NODDI32,X,NaN,NaN,탈락-infarction,NaN,NaN


In [12]:
def append_cols_pid_pdi(base, donor, cols):
    """
    ① PID 로 LEFT JOIN
    ② 남은 NaN 은 AD_MWI_Code ⟷ PDI(부분 문자열)로 보충
    ③ 실제로 생긴 *_donor 컬럼만 정리
    """
    cols = [c for c in cols if c not in ("PID", "PDI")]

    # ── 1) PID 기준 ───────────────────────────────────────────────
    merged = base.merge(
        donor[["PID"] + cols], on="PID", how="left",
        suffixes=("", "_donor")          # 중복시에만 *_donor 생성
    )

    # ── 1-1) 중복 컬럼의 NaN 채우기 ──────────────────────────────
    overlap = [c for c in cols if c in base.columns]
    for c in overlap:
        dcol = f"{c}_donor"
        if dcol in merged.columns:
            merged[c] = merged[c].fillna(merged[dcol])

    # ── 2) AD_MWI_Code ⟷ PDI (부분 문자열) 보충 ───────────────────
    if "AD_MWI_Code" in base.columns and "PDI" in donor.columns:
        d_dict = donor.set_index("PDI")[cols].to_dict("index")
        for pdi_key, vdict in d_dict.items():
            mask = merged["AD_MWI_Code"].astype(str).str.contains(pdi_key, na=False)
            for c, v in vdict.items():
                merged.loc[mask & merged[c].isna(), c] = v

    # ── 3) 실제로 존재하는 *_donor 컬럼만 삭제 ───────────────────
    donor_cols = [c for c in merged.columns if c.endswith("_donor")]
    merged.drop(columns=donor_cols, inplace=True)

    return merged

dfS1 = append_cols_pid_pdi(dfS1, df8641, add_cols_8641)
dfS1 = append_cols_pid_pdi(dfS1, dfDem,  add_cols_8641_dem)

dfS1

,건대분석정보,인원,DDX,AD_MWI_Code,미수코드,논문사용,PID,Name,Age,ChiQSM,...,EC_RT_ParaHippocampus,HFC_RT_Thalamus,IC_RT_Thalamus,EC_RT_Thalamus,HFC_Whole_CC,IC_Whole_CC,EC_Whole_CC,HFC_Whole_Precuneus,IC_Whole_Precuneus,EC_Whole_Precuneus
0,NL(39),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Yea,NaN,NL,EPT2022AD40,NaN,2022 상반기영상,09463703,유길종,74.0,CNL02,...,0.508526,0.863352,0.208503,0.654738,0.545173,0.151938,0.393254,0.678900,0.106224,0.569961
2,No,NaN,NL,EPT2023AD09,NaN,NaN,08787368,이송웅,81.0,CNL03,...,0.447529,0.893360,0.172644,0.720655,0.583949,0.133841,0.450196,0.670014,0.110448,0.559206
3,Yes,1,NL,MWI_CN01,8641.0,"HFC,LFC사용",00145616,강난희,70.0,CNL04,...,0.569473,0.826935,0.271682,0.555182,0.596481,0.212770,0.383751,0.628513,0.125133,0.503758
4,Yes,2,NL,MWI_CN04,8641.0,HFC 사용,02716077,송승숙,62.0,CNL05,...,0.489838,0.761443,0.233806,0.527660,0.482623,0.188198,0.294517,0.592590,0.139421,0.453941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,Yes,9탈락3-2,NaMCI,MWI_CN66(ver2),8641.0,newMWI_CN 27명 추가,00766581,조수연,71.0,CNM25,...,0.436440,0.765378,0.181157,0.584184,0.495296,0.163504,0.331894,0.637991,0.131822,0.508110
202,Yes,10탈락3-3,NaMCI,MWI_CN67(ver2),8641.0,newMWI_CN 27명 추가,09577263,김오심,82.0,CNM26,...,0.583334,0.786152,0.203446,0.582679,0.560845,0.146901,0.414069,0.751461,0.115140,0.640286
203,Yes,13탈락3-4,NaMCI,MWI_CN70(ver2),8641.0,newMWI_CN 27명 추가,02202449,이청자,80.0,CNM27,...,0.494867,0.793464,0.218678,0.574741,0.502785,0.165001,0.337965,0.682435,0.120648,0.561929
204,Yes,탈락-infarction,NaMCI,MREPT2020AD10,NaN,NaN,08276845,조영자,73.0,CNM29,...,0.611423,0.880783,0.222180,0.658885,0.535623,0.164056,0.371696,0.582397,0.128662,0.452970


In [13]:
after = "global CSFV"
orig  = list(pd.read_excel(os.path.join(ALPS_DIR, "Auto_ALPS_index_19-24_0731.xlsx")
                       , sheet_name="Sheet1").columns)  # 원본 순서
new   = [c for c in dfS1.columns if c not in orig]
pos   = dfS1.columns.get_loc(after) + 1
ordered = list(dfS1.columns)
for c in reversed(new):
    ordered.insert(pos, ordered.pop(ordered.index(c)))
dfS1 = dfS1[ordered]

print(len(dfS1.columns))
dfS1

137


,건대분석정보,인원,DDX,AD_MWI_Code,미수코드,논문사용,PID,Name,Age,ChiQSM,...,MR_3DT1_TFE,MR_3DGRASE_MWI,MR_3DGRASE_REF,MREPT_TSE,NODDI,MREPT_3DbFFE,Unnamed: 48,conductivity,MWF,Chi
0,NL(39),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Yea,NaN,NL,EPT2022AD40,NaN,2022 상반기영상,09463703,유길종,74.0,CNL02,...,O,O,O,O,NODDI32,X,NaN,O,O,O
2,No,NaN,NL,EPT2023AD09,NaN,NaN,08787368,이송웅,81.0,CNL03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
3,Yes,1,NL,MWI_CN01,8641.0,"HFC,LFC사용",00145616,강난희,70.0,CNL04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
4,Yes,2,NL,MWI_CN04,8641.0,HFC 사용,02716077,송승숙,62.0,CNL05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,Yes,9탈락3-2,NaMCI,MWI_CN66(ver2),8641.0,newMWI_CN 27명 추가,00766581,조수연,71.0,CNM25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
202,Yes,10탈락3-3,NaMCI,MWI_CN67(ver2),8641.0,newMWI_CN 27명 추가,09577263,김오심,82.0,CNM26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
203,Yes,13탈락3-4,NaMCI,MWI_CN70(ver2),8641.0,newMWI_CN 27명 추가,02202449,이청자,80.0,CNM27,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
204,Yes,탈락-infarction,NaMCI,MREPT2020AD10,NaN,NaN,08276845,조영자,73.0,CNM29,...,O,O,O,O,NODDI32,X,NaN,NaN,탈락-infarction,NaN


In [17]:
analyse_dataframe(dfS1)

,NaN Counts,NaN Ratio (%),Unique Values
건대분석정보,4,1.941748,8
인원,101,49.029126,46
DDX,9,4.368932,5
AD_MWI_Code,9,4.368932,197
미수코드,136,66.019417,1
논문사용,55,26.699029,14
PID,9,4.368932,195
Name,9,4.368932,196
Age,9,4.368932,36
ChiQSM,14,6.796117,192


In [19]:
dfS1.to_excel(os.path.join(ALPS_DIR, 'Auto_ALPS_index_19-24_0801.xlsx')
              , index=False
             )

In [22]:
from pathlib import Path
import shutil
from tqdm import tqdm

src_root = Path(r'D:\JHAhn\8641_conductivity\DTI_ALPS\DTI_ALPS_Result\2_Dementia')
dst_root = Path(r'E:\MR\Dataset')

# 1단계 폴더(예: 개인 ID) 순회
for lvl1 in tqdm([p for p in src_root.iterdir() if p.is_dir()],
                 desc='1st-level', unit='folder'):
    # 2단계 폴더 순회
    for lvl2 in [p for p in lvl1.iterdir() if p.is_dir()]:
        stats_dir = lvl2 / 'Stats'
        if not stats_dir.exists():
            continue

        dest = dst_root / lvl1.name / lvl2.name / 'Stats'
        if dest.exists():
            tqdm.write(f'!!!!! Already exits! SKIPPING . . . : {dest}')
            continue

        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(stats_dir, dest)

1st-level: 100%|█████████████████████████████████████████████████████████████████████| 7/7 [00:20<00:00,  2.89s/folder]


In [21]:
cols = [
    "Measure:volume",
    "Left-Thalamus","Left-Hippocampus","Left-Amygdala","Left-choroid-plexus",
    "Right-Thalamus","Right-Hippocampus","Right-Amygdala","Right-choroid-plexus",
    "CC_Posterior","CC_Mid_Posterior","CC_Central","CC_Mid_Anterior","CC_Anterior",
    "ctx-lh-middletemporal","ctx-lh-parahippocampal","ctx-lh-precuneus","ctx-lh-insula",
    "ctx-rh-middletemporal","ctx-rh-parahippocampal","ctx-rh-precuneus","ctx-rh-insula",
    "EstimatedTotalIntraCranialVol","Left-Lateral-Ventricle","CSF","Right-Lateral-Ventricle",
    "ctx-lh-entorhinal","ctx-rh-entorhinal"
]

# 2단계 하위 폴더의 Stats에서 대상 파일 수집
files = list(src_root.glob(r'*/*/Stats/eTIV_NormalizedBrainVolumeStats.csv'))

rows = []
for fp in tqdm(files, desc='Reading', unit='file'):
    try:
        df = pd.read_csv(fp)
        df = df[cols].copy()  # 지정 열만
    except Exception as e:
        tqdm.write(f'!!!! Skipping : No Col(s) | File problem(s) : {fp} → {e}')
        continue

    # 개인 ID = <lvl2> 폴더명(…\2_Dementia\<lvl1>\<lvl2>\Stats\file)
    subject = fp.parent.parent.name
    if subject.endswith('_Results'):
        subject = subject[:-len('_Results')]
    df.insert(0, 'Subject', subject)

    # 수치형 변환 (Measure:volume 제외)
    num_cols = [c for c in cols if c != 'Measure:volume']
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

    rows.append(df)

if not rows:
    raise SystemExit('Not available datasets')

merged = pd.concat(rows, ignore_index=True)
merged

Reading: 100%|████████████████████████████████████████████████████████████████████| 269/269 [00:02<00:00, 112.82file/s]


,Subject,Measure:volume,Left-Thalamus,Left-Hippocampus,Left-Amygdala,Left-choroid-plexus,Right-Thalamus,Right-Hippocampus,Right-Amygdala,Right-choroid-plexus,...,ctx-rh-middletemporal,ctx-rh-parahippocampal,ctx-rh-precuneus,ctx-rh-insula,EstimatedTotalIntraCranialVol,Left-Lateral-Ventricle,CSF,Right-Lateral-Ventricle,ctx-lh-entorhinal,ctx-rh-entorhinal
0,MWI_CN01_KNH_NODDI32,MWI_CN01_KNH_NODDI32,0.636927,0.320623,0.151001,0.055123,0.621512,0.349313,0.168211,0.057446,...,1.153423,0.145693,0.949497,0.551550,100.0,1.369492,0.136431,1.042649,0.188636,0.152508
1,MWI_CN04_SSS_NODDI32,MWI_CN04_SSS_NODDI32,0.809188,0.428989,0.201388,0.047482,0.767186,0.411734,0.217380,0.062679,...,1.370355,0.198570,0.981591,0.529487,100.0,0.467099,0.083311,0.579552,0.237150,0.215680
2,MWI_CN05_LSS_NODDI32,MWI_CN05_LSS_NODDI32,0.587279,0.355454,0.160146,0.079509,0.556028,0.396543,0.156755,0.071776,...,1.128888,0.162656,0.730390,0.536571,100.0,1.008765,0.145763,0.864571,0.182624,0.175534
3,MWI_CN06_KDS_NODDI32,MWI_CN06_KDS_NODDI32,0.478851,0.285343,0.124687,0.057938,0.454111,0.293534,0.141689,0.052879,...,0.892694,0.126733,0.639348,0.434330,100.0,1.240260,0.091902,1.122332,0.140422,0.141195
4,MWI_CN07_KSB_NODDI32,MWI_CN07_KSB_NODDI32,0.612925,0.354023,0.142393,0.050115,0.590602,0.364602,0.164676,0.058506,...,1.075945,0.141411,0.716557,0.478708,100.0,1.080631,0.090091,1.042915,0.196452,0.192596
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,EPT2024AD09_CSO_NODDI32,EPT2024AD09_CSO_NODDI32,0.388010,0.211934,0.081108,0.045172,0.401554,0.207772,0.070913,0.053696,...,0.778733,0.063002,0.559156,0.311999,100.0,2.167420,0.169744,1.343576,0.130880,0.082595
265,EPT2024AD10_KCJ_NODDI32,EPT2024AD10_KCJ_NODDI32,0.668859,0.344226,0.134545,0.089703,0.631830,0.379854,0.161534,0.080121,...,1.089978,0.179310,0.706567,0.482590,100.0,1.155754,0.136460,1.046001,0.180641,0.187852
266,EPT2024AD11_KHJ_NODDI32,EPT2024AD11_KHJ_NODDI32,0.436845,0.234100,0.117182,0.068399,0.326249,0.261052,0.124230,0.053842,...,0.811449,0.113046,0.510056,0.293556,100.0,1.648145,0.101376,1.362230,0.142228,0.136102
267,EPT2024AD12_LDS_NODDI32,EPT2024AD12_LDS_NODDI32,0.393478,0.208205,0.078369,0.086453,0.343730,0.216806,0.094104,0.085581,...,0.870929,0.085079,0.465506,0.423018,100.0,1.663795,0.159006,1.314226,0.099974,0.105460


In [22]:
assert merged['Subject'].astype(str).eq(merged['Measure:volume'].astype(str)).all(), \
       "!!!! Mismatch Found : Between col 'Subject' & 'Measure:volume' in rows"

In [23]:
merged.drop(columns='Subject', inplace=True)

In [26]:
# [1]
if 'EstimatedTotalIntraCranialVol' in merged.columns:
    order = list(merged.columns)
    order.remove('EstimatedTotalIntraCranialVol')
    mv_idx = order.index('Measure:volume') + 1
    order.insert(mv_idx, 'EstimatedTotalIntraCranialVol')
else:
    order = list(merged.columns)

# [2]
cc_cols = [c for c in merged.columns if c.startswith('CC_')]
if cc_cols:
    merged['CorpusCallosum'] = merged[cc_cols].sum(axis=1, min_count=1)
    first_cc_idx = min(order.index(c) for c in cc_cols)
    if 'CorpusCallosum' in order:
        order.remove('CorpusCallosum')
    order.insert(first_cc_idx, 'CorpusCallosum')
merged = merged[order]

# [3]
name_map = {c: c for c in merged.columns}
name_map['Measure:volume'] = 'PDI'
name_map['TIV'] = 'Normalised_TIV'
name_map['Left-Thalamus'] = 'LT_Thalamus'
name_map['Left-Hippocampus'] = 'LT_Hippocampus'
name_map['Left-Amygdala'] = 'LT_Amygdala'
name_map['Left-choroid-plexus'] = 'LT_ChoroidPlexus'
name_map['Right-Thalamus'] = 'RT_Thalamus'
name_map['Right-Hippocampus'] = 'RT_Hippocampus'
name_map['Right-Amygdala'] = 'RT_Amygdala'
name_map['Right-choroid-plexus'] = 'RT_ChoroidPlexus'
name_map['CC_Posterior'] = 'Pos_CC'
name_map['CC_Mid_Posterior'] = 'MidPos_CC'
name_map['CC_Central'] = 'Central_CC'
name_map['CC_Mid_Anterior'] = 'MidAnt_CC'
name_map['CC_Anterior'] = 'Ant_CC'
name_map['ctx-lh-middletemporal'] = 'LT_MiddleTemporal_CTX'
name_map['ctx-lh-parahippocampal'] = 'LT_ParaHippocampal_CTX'
name_map['ctx-lh-precuneus'] = 'LT_Precuneus_CTX'
name_map['ctx-lh-insula'] = 'LT_Insula_CTX'
name_map['ctx-rh-middletemporal'] = 'RT_MiddleTemporal_CTX'
name_map['ctx-rh-parahippocampal'] = 'RT_ParaHippocampal_CTX'
name_map['ctx-rh-precuneus'] = 'RT_Precuneus_CTX'
name_map['ctx-rh-insula'] = 'RT_Insula_CTX'
name_map['EstimatedTotalIntraCranialVol'] = 'TIV'
name_map['Left-Lateral-Ventricle'] = 'LT_Lat_Ventricle'
name_map['CSF'] = 'CSF'
name_map['Right-Lateral-Ventricle'] = 'RT_Lat_Ventricle'
name_map['ctx-lh-entorhinal'] = 'LT_Entorhinal_CTX'
name_map['ctx-rh-entorhinal'] = 'RT_Entorhinal_CTX'
merged.rename(columns=name_map, inplace=True)

merged

,PDI,Normalised_TIV,LT_Thalamus,LT_Hippocampus,LT_Amygdala,LT_ChoroidPlexus,RT_Thalamus,RT_Hippocampus,RT_Amygdala,RT_ChoroidPlexus,...,LT_Insula_CTX,RT_MiddleTemporal_CTX,RT_ParaHippocampal_CTX,RT_Precuneus_CTX,RT_Insula_CTX,LT_Lat_Ventricle,CSF,RT_Lat_Ventricle,LT_Entorhinal_CTX,RT_Entorhinal_CTX
0,MWI_CN01_KNH_NODDI32,100.0,0.636927,0.320623,0.151001,0.055123,0.621512,0.349313,0.168211,0.057446,...,0.528590,1.153423,0.145693,0.949497,0.551550,1.369492,0.136431,1.042649,0.188636,0.152508
1,MWI_CN04_SSS_NODDI32,100.0,0.809188,0.428989,0.201388,0.047482,0.767186,0.411734,0.217380,0.062679,...,0.506306,1.370355,0.198570,0.981591,0.529487,0.467099,0.083311,0.579552,0.237150,0.215680
2,MWI_CN05_LSS_NODDI32,100.0,0.587279,0.355454,0.160146,0.079509,0.556028,0.396543,0.156755,0.071776,...,0.487687,1.128888,0.162656,0.730390,0.536571,1.008765,0.145763,0.864571,0.182624,0.175534
3,MWI_CN06_KDS_NODDI32,100.0,0.478851,0.285343,0.124687,0.057938,0.454111,0.293534,0.141689,0.052879,...,0.435603,0.892694,0.126733,0.639348,0.434330,1.240260,0.091902,1.122332,0.140422,0.141195
4,MWI_CN07_KSB_NODDI32,100.0,0.612925,0.354023,0.142393,0.050115,0.590602,0.364602,0.164676,0.058506,...,0.511546,1.075945,0.141411,0.716557,0.478708,1.080631,0.090091,1.042915,0.196452,0.192596
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,EPT2024AD09_CSO_NODDI32,100.0,0.388010,0.211934,0.081108,0.045172,0.401554,0.207772,0.070913,0.053696,...,0.303329,0.778733,0.063002,0.559156,0.311999,2.167420,0.169744,1.343576,0.130880,0.082595
265,EPT2024AD10_KCJ_NODDI32,100.0,0.668859,0.344226,0.134545,0.089703,0.631830,0.379854,0.161534,0.080121,...,0.453843,1.089978,0.179310,0.706567,0.482590,1.155754,0.136460,1.046001,0.180641,0.187852
266,EPT2024AD11_KHJ_NODDI32,100.0,0.436845,0.234100,0.117182,0.068399,0.326249,0.261052,0.124230,0.053842,...,0.316960,0.811449,0.113046,0.510056,0.293556,1.648145,0.101376,1.362230,0.142228,0.136102
267,EPT2024AD12_LDS_NODDI32,100.0,0.393478,0.208205,0.078369,0.086453,0.343730,0.216806,0.094104,0.085581,...,0.406796,0.870929,0.085079,0.465506,0.423018,1.663795,0.159006,1.314226,0.099974,0.105460


In [27]:
analyse_dataframe(merged)

,NaN Counts,NaN Ratio (%),Unique Values
PDI,0,0.0,269
Normalised_TIV,0,0.0,1
LT_Thalamus,0,0.0,269
LT_Hippocampus,0,0.0,269
LT_Amygdala,0,0.0,269
LT_ChoroidPlexus,0,0.0,269
RT_Thalamus,0,0.0,269
RT_Hippocampus,0,0.0,269
RT_Amygdala,0,0.0,269
RT_ChoroidPlexus,0,0.0,269


In [28]:
os.makedirs(out, exist_ok=True)
merged.to_excel(os.path.join(out, 'Normalised_BrainVolume.xlsx'), index=False)

In [30]:
dfg = pd.read_excel(os.path.join(out, 'Auto_ALPS_index_19-24_0801.xlsx'))
dfg['AD_MWI_Code'] = (dfg['AD_MWI_Code']
                        .astype(str)
                        .str.replace(r'\s*\([^)]*\)\s*$', '', regex=True)
                        .str.strip())
dfg

,건대분석정보,인원,DDX,AD_MWI_Code,미수코드,논문사용,PID,Name,Age,ChiQSM,...,MR_3DT1_TFE,MR_3DGRASE_MWI,MR_3DGRASE_REF,MREPT_TSE,NODDI,MREPT_3DbFFE,Unnamed: 48,conductivity,MWF,Chi
0,NL(39),NaN,NaN,nan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Yea,NaN,NL,EPT2022AD40,NaN,2022 상반기영상,9463703.0,유길종,74.0,CNL02,...,O,O,O,O,NODDI32,X,NaN,O,O,O
2,No,NaN,NL,EPT2023AD09,NaN,NaN,8787368.0,이송웅,81.0,CNL03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
3,Yes,1,NL,MWI_CN01,8641.0,"HFC,LFC사용",145616.0,강난희,70.0,CNL04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
4,Yes,2,NL,MWI_CN04,8641.0,HFC 사용,2716077.0,송승숙,62.0,CNL05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,Yes,9탈락3-2,NaMCI,MWI_CN66,8641.0,newMWI_CN 27명 추가,766581.0,조수연,71.0,CNM25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
202,Yes,10탈락3-3,NaMCI,MWI_CN67,8641.0,newMWI_CN 27명 추가,9577263.0,김오심,82.0,CNM26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
203,Yes,13탈락3-4,NaMCI,MWI_CN70,8641.0,newMWI_CN 27명 추가,2202449.0,이청자,80.0,CNM27,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,O,O,O
204,Yes,탈락-infarction,NaMCI,MREPT2020AD10,NaN,NaN,8276845.0,조영자,73.0,CNM29,...,O,O,O,O,NODDI32,X,NaN,NaN,탈락-infarction,NaN


In [31]:
df = dfg
for column in df.columns:
    print(f"Col name: {column}")
    print(f"Data type: {df[column].dtype}")
    print(f"Unique values: {len(df[column].unique())}")
    print(df[column].unique())
    print('-'*40)

Col name: 건대분석정보
Data type: object
Unique values: 9
['NL(39)' 'Yea' 'No' 'Yes' nan 'SaMCI(41)' 'MaMCI(36)' 'AD (35)'
 'NaMCI(20)']
----------------------------------------
Col name: 인원
Data type: object
Unique values: 47
[nan 1 2 3 4 5 10 17 18 19 20 21 22 8 9 11 14 15 25 6 12 13 '탈락' 27 24 16
 7 '점수미달' 'MREPT2019AD10' 23 28 '탈락-hydrocephalus' 'MREPT2019AD09'
 'MREPT2019AD13' '탈락-SAH' '4탈락-' 'MREPT2019AD11' '7탈락1-2' '12탈락1-5'
 '14탈락1-7' '7탈락2-1' '19탈락2-2' '9탈락3-2' '10탈락3-3' '13탈락3-4' '탈락-infarction'
 '탈락-venous anomaly']
----------------------------------------
Col name: DDX
Data type: object
Unique values: 6
[nan 'NL' 'SaMCI' 'MaMCI' 'AD' 'NaMCI']
----------------------------------------
Col name: AD_MWI_Code
Data type: object
Unique values: 198
['nan' 'EPT2022AD40' 'EPT2023AD09' 'MWI_CN01' 'MWI_CN04' 'MWI_CN05'
 'MWI_CN06' 'MWI_CN07' 'MWI_CN12' 'MWI_CN19' 'MWI_CN20' 'MWI_CN21'
 'MWI_CN22' 'MWI_CN23' 'MWI_CN24' 'MWI_CN33' 'MWI_CN35' 'MWI_CN38'
 'MWI_CN39' 'MWI_CN40' 'MWI_CN41' 'MWI_CN

In [33]:
# merged = pd.read_excel(os.path.join(out, 'Normalised_BrainVolume.xlsx'))
merged['PDI'] = merged['PDI'].astype(str).str.strip()

def match_pdi(code):
    if not isinstance(code, str):
        return None
    code = code.strip()
    if code == '' or code.lower() == 'nan':
        return None
    s = merged['PDI'].astype(str)
    # 1 : Equal conditions
    eq = s[s == code]
    if len(eq) >= 1:
        return eq.iloc[0]
    # 2 : Contain conditions
    contains = s[s.str.contains(re.escape(code), na=False)]
    if len(contains) >= 1:
        return contains.iloc[0]

    return None

dfg['Matched_PDI'] = dfg['AD_MWI_Code'].apply(match_pdi)
res = dfg.merge(merged, left_on='Matched_PDI', right_on='PDI', how='left')

res

,건대분석정보,인원,DDX,AD_MWI_Code,미수코드,논문사용,PID,Name,Age,ChiQSM,...,LT_Insula_CTX,RT_MiddleTemporal_CTX,RT_ParaHippocampal_CTX,RT_Precuneus_CTX,RT_Insula_CTX,LT_Lat_Ventricle,CSF,RT_Lat_Ventricle,LT_Entorhinal_CTX,RT_Entorhinal_CTX
0,NL(39),NaN,NaN,nan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Yea,NaN,NL,EPT2022AD40,NaN,2022 상반기영상,9463703.0,유길종,74.0,CNL02,...,0.437793,1.012329,0.158690,0.708591,0.428610,1.165589,0.110753,0.980698,0.189083,0.167442
2,No,NaN,NL,EPT2023AD09,NaN,NaN,8787368.0,이송웅,81.0,CNL03,...,0.346210,0.778912,0.115093,0.610142,0.367369,1.740043,0.103345,1.501742,0.114483,0.142610
3,Yes,1,NL,MWI_CN01,8641.0,"HFC,LFC사용",145616.0,강난희,70.0,CNL04,...,0.528590,1.153423,0.145693,0.949497,0.551550,1.369492,0.136431,1.042649,0.188636,0.152508
4,Yes,2,NL,MWI_CN04,8641.0,HFC 사용,2716077.0,송승숙,62.0,CNL05,...,0.506306,1.370355,0.198570,0.981591,0.529487,0.467099,0.083311,0.579552,0.237150,0.215680
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,Yes,9탈락3-2,NaMCI,MWI_CN66,8641.0,newMWI_CN 27명 추가,766581.0,조수연,71.0,CNM25,...,0.450704,1.096321,0.185040,0.878919,0.487818,1.460113,0.137387,1.243020,0.195444,0.205491
202,Yes,10탈락3-3,NaMCI,MWI_CN67,8641.0,newMWI_CN 27명 추가,9577263.0,김오심,82.0,CNM26,...,0.255111,0.742859,0.117688,0.454969,0.303533,1.391899,0.123960,1.251832,0.069831,0.113590
203,Yes,13탈락3-4,NaMCI,MWI_CN70,8641.0,newMWI_CN 27명 추가,2202449.0,이청자,80.0,CNM27,...,0.318675,0.701914,0.085252,0.510614,0.299052,1.049790,0.120790,0.835080,0.157372,0.105387
204,Yes,탈락-infarction,NaMCI,MREPT2020AD10,NaN,NaN,8276845.0,조영자,73.0,CNM29,...,0.390007,1.055012,0.174523,0.819847,0.434248,1.437341,0.142681,1.168819,0.165392,0.167523


In [35]:
mask = res['PID'].notna()
res.loc[mask, 'PID'] = (res.loc[mask, 'PID']
                          .astype(str)
                          .str.replace(r'\.0$', '', regex=True)   # 끝의 .0 제거
                          .str.extract(r'(\d+)', expand=False)     # 숫자만 추출
                          .str.zfill(8))                           # 8자리로 0 채움
res

,건대분석정보,인원,DDX,AD_MWI_Code,미수코드,논문사용,PID,Name,Age,ChiQSM,...,LT_Insula_CTX,RT_MiddleTemporal_CTX,RT_ParaHippocampal_CTX,RT_Precuneus_CTX,RT_Insula_CTX,LT_Lat_Ventricle,CSF,RT_Lat_Ventricle,LT_Entorhinal_CTX,RT_Entorhinal_CTX
0,NL(39),NaN,NaN,nan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Yea,NaN,NL,EPT2022AD40,NaN,2022 상반기영상,09463703,유길종,74.0,CNL02,...,0.437793,1.012329,0.158690,0.708591,0.428610,1.165589,0.110753,0.980698,0.189083,0.167442
2,No,NaN,NL,EPT2023AD09,NaN,NaN,08787368,이송웅,81.0,CNL03,...,0.346210,0.778912,0.115093,0.610142,0.367369,1.740043,0.103345,1.501742,0.114483,0.142610
3,Yes,1,NL,MWI_CN01,8641.0,"HFC,LFC사용",00145616,강난희,70.0,CNL04,...,0.528590,1.153423,0.145693,0.949497,0.551550,1.369492,0.136431,1.042649,0.188636,0.152508
4,Yes,2,NL,MWI_CN04,8641.0,HFC 사용,02716077,송승숙,62.0,CNL05,...,0.506306,1.370355,0.198570,0.981591,0.529487,0.467099,0.083311,0.579552,0.237150,0.215680
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,Yes,9탈락3-2,NaMCI,MWI_CN66,8641.0,newMWI_CN 27명 추가,00766581,조수연,71.0,CNM25,...,0.450704,1.096321,0.185040,0.878919,0.487818,1.460113,0.137387,1.243020,0.195444,0.205491
202,Yes,10탈락3-3,NaMCI,MWI_CN67,8641.0,newMWI_CN 27명 추가,09577263,김오심,82.0,CNM26,...,0.255111,0.742859,0.117688,0.454969,0.303533,1.391899,0.123960,1.251832,0.069831,0.113590
203,Yes,13탈락3-4,NaMCI,MWI_CN70,8641.0,newMWI_CN 27명 추가,02202449,이청자,80.0,CNM27,...,0.318675,0.701914,0.085252,0.510614,0.299052,1.049790,0.120790,0.835080,0.157372,0.105387
204,Yes,탈락-infarction,NaMCI,MREPT2020AD10,NaN,NaN,08276845,조영자,73.0,CNM29,...,0.390007,1.055012,0.174523,0.819847,0.434248,1.437341,0.142681,1.168819,0.165392,0.167523


In [36]:
os.makedirs(out, exist_ok=True)
res.to_excel(os.path.join(out, 'Auto_ALPS_index_19-24_0805.xlsx'), index=False)